# Visual Inspection Training Program
Use this notebook to practice visual inspection. Also use it to flag anchors that you disagree with Julia's classification

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from time import perf_counter, sleep
from datetime import datetime

import os
import glob # get path names
from pathlib import Path 

import pandas as pd
from astropy.table import Table
from astropy.io import fits

import astropy.visualization as vis # for image enhancements
from scipy.ndimage import gaussian_filter

import ipywidgets as widgets # for buttons
from IPython.display import display, clear_output # for display functions

In [2]:
# Load training set
anchors = Table.read("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv")

# Map main type to classification
main_type_map = {
    20: "E",
    10: "S",
    0: "L",
    -5: "I"
}

# Normalize class labels
anchors["trainer_class"] = [main_type_map[m] for m in anchors["Main_type"]]

In [3]:
# Track program user
USERNAME = "QuillanIMR"

# Initialize statistics
total = 0
correct = 0
current_galaxy = None
start_time = None

# Random order
remaining_indices = np.random.permutation(len(anchors))
current_position = 0

# Prepare image flag system
flagged = []
last_result = None

# Session indicator
session_running = True

In [4]:
# Image enhancer
def stretch_band(band, mode="asinh"):
    band = np.nan_to_num(band)
    if mode == "asinh":
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.AsinhStretch())
    elif mode == "log":
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.LogStretch())
    else:
        norm = vis.ImageNormalize(band, interval=vis.PercentileInterval(99.5), stretch=vis.LinearStretch())
    return norm(band)
    
# Image loader
def load_image(path, stretch="asinh"):

    # open fits file
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)

    # Assign image data
    img = np.transpose(data, (1,2,0))
    g = img[:,:,0]
    r = img[:,:,1]
    z = img[:,:,2]

    # Add image stretch
    g_str = stretch_band(g, stretch)
    r_str = stretch_band(r, stretch)
    z_str = stretch_band(z, stretch)

    # Combine band layers by depth and bind pixel values
    rgb = np.dstack([z_str, r_str, g_str])
    rgb = np.clip(rgb,0,1)

    # Combine bands for a grayscale image
    gray = (0.5*r_str + 0.3*z_str + 0.2*g_str)
    gray = np.clip(gray,0,1)

    # Use Lupton technique
    lupton = vis.make_lupton_rgb(z, r, g, stretch=0.5, Q=10)

    # Create Gaussian blur   
    smooth = gaussian_filter(gray, sigma=4)

    # Isolate fine details with unsharp mask
    unsharp = gray - smooth    
    unsharp -= unsharp.min()
    unsharp /= unsharp.max()

    return rgb, lupton, gray, unsharp, g_str, r_str, z_str

# Load legacy survey jpgs
def load_jpg(row):
    # Get galaxy id
    tid = row["ref_id"]

    patternM = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Model/{tid}_*.jpg"
    matchesM = glob.glob(patternM)
    patternR = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Residual/{tid}_*.jpg"
    matchesR = glob.glob(patternR)
    patternI = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Image/{tid}_*.jpg"
    matchesI = glob.glob(patternI)

    model = mpimg.imread(matchesM[0])
    residual = mpimg.imread(matchesR[0])
    image = mpimg.imread(matchesI[0])
 
    return model, residual, image

# Get cutout path
def cutout_path(row):
    # Get galaxy id
    tid = row["ref_id"]

    # Search cutout folders for anchor images
    for folder in ["Elliptical", "Spiral", "Lenticular", "Irregular"]:
        
        pattern = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/{folder}/{tid}_grz_152_*.fits"
        matches = glob.glob(pattern)
        
        if matches:
            return matches[0]
           
    raise FileNotFoundError(tid) # If file is missing, raise error

In [5]:
# Save information about previous galaxy for inspection
def flag_previous(_):

    global last_result

    # Ensure previous galaxy exists
    if last_result is None:
        return
        
    # Ignore previously flagged galaxies
    if any(f["ref_id"] == last_result["ref_id"] for f in flagged):
        return

    # Save flagged galaxies
    flagged.append(last_result.copy())

    flag_path = Path(f"/pscratch/sd/q/qshimp/VI_training/savefiles/{USERNAME}_flagged.csv")
    
    new_flag = pd.DataFrame([last_result])    
    new_flag.to_csv(flag_path, mode="a", header=not flag_path.exists(), index=False)
    
    # Display flagging confirmation  
    with info_out:    
        info_out.clear_output(wait=True)    
        pct = 100 * correct / total if total else 0    
        print(f"Score: {correct}/{total} ({pct:.1f}%)")
        print(f"Remaining: {len(remaining_indices)-current_position}")
        print()
        print(f"Flagged galaxy {last_result['ref_id']}")

In [6]:
# Create displays for images and interface
image_out = widgets.Output()
info_out = widgets.Output()

# Create stretch selector
stretch_selector = widgets.Dropdown(
    options=[("Asinh (recommended)", "asinh"), ("Log", "log"), ("Linear", "linear")],
    value="asinh",
    description="Stretch:"
)

# Create view selector
view_selector = widgets.Dropdown(
    options=[
        "RGB",
        "Lupton RGB",
        "Grayscale",
        "Unsharp Mask",
        "RGB + Grayscale",
        "RGB + Grayscale + Unsharp",
        "Image + Model + Residuals",
        "6-view", 
        "g band",
        "r band",
        "z band"
    ],
    value="RGB + Grayscale + Unsharp",
    description="View:"
)

# Create buttons
btn_E = widgets.Button(description="Elliptical")
btn_L = widgets.Button(description="Lenticular")
btn_S = widgets.Button(description="Spiral")
btn_I = widgets.Button(description="Irregular")
btn_skip = widgets.Button(description="Skip")
btn_stop = widgets.Button(description="Stop")
btn_flag_prev = widgets.Button(description="Flag Previous", button_style="warning")

# Create functional interface
controls = widgets.HBox([stretch_selector, view_selector])
buttons = widgets.HBox([btn_E, btn_L, btn_S, btn_I, btn_skip, btn_stop, btn_flag_prev])
display(controls, image_out, info_out, buttons)

Output()

Output()

In [7]:
# Function to display current galaxy
def display_galaxy():

    # Load image data 
    rgb, lupton, gray, unsharp, g, r, z = load_image( cutout_path(current_galaxy), stretch=stretch_selector.value)

    model, residual, image = load_jpg(current_galaxy)

    # Display image
    with image_out:

        # Reset image display
        image_out.clear_output(wait=True)
        view = view_selector.value

        # RGB
        if view == "RGB":

            plt.figure(figsize=(6,6))
            plt.imshow(rgb, origin="lower")
            plt.title("RGB")
            plt.axis("off")

        # Lupton RGB
        elif view == "Lupton RGB":

            plt.figure(figsize=(6,6))
            plt.imshow(lupton, origin="lower")
            plt.title("Lupton RGB")
            plt.axis("off")

        # Grayscale
        elif view == "Grayscale":

            plt.figure(figsize=(6,6))
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

        # Unsharp Mask
        elif view == "Unsharp Mask":

            plt.figure(figsize=(6,6))
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp Mask")
            plt.axis("off")

        # RGB + Grayscale
        elif view == "RGB + Grayscale":

            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.imshow(rgb, origin="lower")
            plt.title("RGB")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

        # RGB + Grayscale + Unsharp
        elif view == "RGB + Grayscale + Unsharp":

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(rgb, origin="lower")
            plt.title("RGB")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp")
            plt.axis("off")

        # Image + Model + Residuals
        elif view == "Image + Model + Residuals":

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(image)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(model)
            plt.title("Model")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(residual)
            plt.title("Residuals")
            plt.axis("off")

        # Image + Model + Residuals + RGB + Grayscale + Unsharp
        elif view == "6-view":

            plt.figure(figsize=(15,10))

            plt.subplot(2,3,1)
            plt.imshow(image)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(2,3,2)
            plt.imshow(model)
            plt.title("Model")
            plt.axis("off")

            plt.subplot(2,3,3)
            plt.imshow(residual)
            plt.title("Residuals")
            plt.axis("off")

            plt.subplot(2,3,4)
            plt.imshow(rgb, origin="lower")
            plt.title("RGB")
            plt.axis("off")

            plt.subplot(2,3,5)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

            plt.subplot(2,3,6)
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp")
            plt.axis("off")
            
        # g band
        elif view == "g band":

            plt.figure(figsize=(6,6))
            plt.imshow(g, origin="lower", cmap="gray")
            plt.title("g band")
            plt.axis("off")

        # r band
        elif view == "r band":

            plt.figure(figsize=(6,6))
            plt.imshow(r, origin="lower", cmap="gray")
            plt.title("r band")
            plt.axis("off")

        # z band
        elif view == "z band":

            plt.figure(figsize=(6,6))
            plt.imshow(z, origin="lower", cmap="gray")
            plt.title("z band")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

In [8]:
# Get next galaxy
def new_galaxy():

    global current_galaxy, start_time, current_position

    # If visual inspection is finished
    if current_position >= len(remaining_indices):
        print("All galaxies completed!")
        save()
        return

    # Index through anchor set
    idx = remaining_indices[current_position]
    current_position += 1
    current_galaxy = anchors[idx]

    # Start timer
    start_time = perf_counter()

    # Display image
    display_galaxy()

    # Clear image display and output score display
    with info_out:

        info_out.clear_output(wait=True)

        pct = 100 * correct / total if total else 0

        print(f"Score: {correct}/{total} ({pct:.1f}%)")
        print(f"Remaining: {len(remaining_indices)-current_position}")
        print("Classify the galaxy:")

In [9]:
# Handle user input
def handle_answer(choice):
    global total, correct, user_results, last_result 

    timestamp = datetime.now().isoformat(timespec="seconds")
    
    # Skip if no galaxy
    if current_galaxy is None:
        return

    # Calculate response time
    response_time = perf_counter() - start_time

    # Get true class
    true = current_galaxy["trainer_class"]

    # Skip galaxy
    if choice == "skip":
        new_galaxy()
        return

    # Stop session
    if choice == "stop":
        global session_running
    
        # Disable buttons
        session_running = False
        ui_enabled(False)
    
        # Enable start button
        btn_stop.description = "Start"
        btn_stop.button_style = "success"
    
        # Pause message
        with info_out:
            info_out.clear_output(wait=True)
            print("Session paused. Click Start to resume.")
    
        return

    # Get boolean of choice
    is_correct = (choice == true)

    # Disable flag button if correct
    btn_flag_prev.disabled = is_correct

    # Track overall statistics
    total += 1
    if is_correct:
        correct += 1

    # Append statistics to savefile
    new_row = pd.DataFrame([{
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "username": USERNAME,
        "target_id": int(current_galaxy["ref_id"]),
        "true_class": true,
        "user_class": choice,
        "correct": is_correct,
        "response_time": response_time
    }])
    
    path = Path(f"/pscratch/sd/q/qshimp/VI_training/savefiles/{USERNAME}.csv")
    
    new_row.to_csv(
        path,
        mode="a",
        header=not path.exists(),
        index=False
    )

    # Display information for user after choice
    with info_out:
        info_out.clear_output(wait=True)
        print("CORRECT! :)" if is_correct else "INCORRECT :(")
        print(f"True: {true} | You: {choice}")
        print(f"Time: {response_time:.2f}s")
        print(f"Score: {correct}/{total}")
    
    # Keep feedback visible
    sleep(0.5)

    # Store information of this galaxy
    last_result = {
        "ref_id": int(current_galaxy["ref_id"]),
        "RA": float(current_galaxy["ra"]),
        "DEC": float(current_galaxy["dec"]),
        "true_class": true,
        "your_class": choice,
        "path": cutout_path(current_galaxy),
        "correct": is_correct,
        "notes": ""
    }

    # Get new galaxy
    new_galaxy()

In [10]:
# Refresh if selector is used
def refresh(change):
    if current_galaxy is not None:
        display_galaxy()
        
stretch_selector.observe(refresh, names="value")
view_selector.observe(refresh, names="value")

# Enable/disable buttons
def ui_enabled(state):
    for btn in [btn_E, btn_L, btn_S, btn_I, btn_skip, btn_flag_prev]:
        btn.disabled = not state

    stretch_selector.disabled = not state
    view_selector.disabled = not state

# Create stop/start button 
def toggle_session(_):
    global session_running

    # Stop the session
    if session_running:
        handle_answer("stop")

    # Start the session
    else:
        session_running = True
        ui_enabled(True)

        # Change button from start to stop
        btn_stop.description = "Stop"
        btn_stop.button_style = "danger"

        # Output resume message
        with info_out:
            info_out.clear_output(wait=True)
            pct = 100 * correct / total if total else 0
            print("Session resumed.")
            print(f"Score: {correct}/{total} ({pct:.1f}%)")
            print(f"Remaining: {len(remaining_indices)-current_position}")

In [11]:
# Add button functionality
btn_E.on_click(lambda x: handle_answer("E"))
btn_L.on_click(lambda x: handle_answer("L"))
btn_S.on_click(lambda x: handle_answer("S"))
btn_I.on_click(lambda x: handle_answer("I"))
btn_skip.on_click(lambda x: handle_answer("skip"))
btn_stop.on_click(toggle_session)
btn_flag_prev.on_click(flag_previous)

In [12]:
# Initialize session
new_galaxy()